In [4]:
import numpy as np

#BODY-----------------------------------------
class Body:
    def __init__(self, m=1.0, r=(0.0, 0.0), v=(0.0, 0.0), radius=0.0,
                    fixed=False, name=""):
        self.m, self.radius = float(m), float(radius)
        self.fixed = bool(fixed)
        self.name = name
        
        self.r = np.array(r, dtype=float)
        self.v = np.array(v, dtype=float)
        self.f = np.zeros(2)

    @property
    def inv_m(self):
        return 0.0 if self.fixed else 1.0 / self.m

    def clear(self):
        self.f[:] = 0.0

    def add_force(self, F):
        self.f += np.asarray(F, dtype=float)

    def ke(self):
        return 0.5 * self.m * self.v.dot(self.v)

    def p(self):
        return self.m * self.v

    def __repr__(self):
        return f"Body({self.name}, m={self.m}, r={self.r}, v={self.v})"



#FORCE-------------------------------------------------
class Force:
    def potential_energy(self, world):
        return 0.0

class Gravity(Force):
    def __init__(self, g=9.81):
        self.gmag = g
        self.g    = np.array([0.0, -g])

    def apply(self, world):
        for b in world.bodies:
            if not b.fixed:
                b.add_force(b.m * self.g)

    def potential_energy(self, world):
        return sum(b.m * self.gmag * b.r[1] for b in world.bodies if not b.fixed)


#INTEGRATOR
class EulerCromer:
    name= "Euler Cromer"
    def step(self, world, dt):
        world.compute_forces()
        for b in world.bodies:
            if b.fixed:
                continue
            b.v += (b.f*b.inv_m)*dt
            b.r += b.v*dt

class Euler:
    name = "Euler"
    def step(self, world, dt):
        world.compute_forces()
        for b in world.bodies:
            if b.fixed:
                continue
            a = b.f * b.inv_m
            b.r     += b.v * dt 
            b.v     += a * dt



#WORLD-------------------------------------------
class World:
    def __init__(self, integrator=None, forces=None, bodies=None, constraints=None):
        self.bodies      = list(bodies or [])
        self.forces      = list(forces or [])
        self.constraints = list(constraints or [])
        self.integrator  = integrator
        self.t           = 0.0
        self.recorders   = {}
        self.history     = {}

    def add(self, body):
        self.bodies.append(body); return body

    def add_force(self, force):
        self.forces.append(force); return force

    def add_constraint(self, c):
        self.constraints.append(c); return c

    def compute_forces(self):
        for b in self.bodies:
            b.clear()
        for f in self.forces:
            f.apply(self)                  

    def step(self, dt):
        self.integrator.step(self, dt)      
        for c in self.constraints:          
            c.resolve(self)
        self.t += dt                              

    
    def kinetic_energy(self):
        return sum(b.ke() for b in self.bodies)
    def potential_energy(self):
        return sum(f.potential_energy(self) for f in self.forces)
    def total_energy(self):
        return self.kinetic_energy() + self.potential_energy()
    def momentum(self):
        return sum((b.p() for b in self.bodies), np.zeros(2))

    
    def record(self, name, fn):
        self.recorders[name] = fn
        self.history[name]   = []

    def capture(self):
        for k, fn in self.recorders.items():
            self.history[k].append(fn(self))

    def run(self, dt, t_end):
        self.capture()
        for _ in range(int(round(t_end / dt))):
            self.step(dt)
            self.capture()
        return {k: np.array(v) for k, v in self.history.items()}

In [5]:
speed = 20.0
angle_degrees = 50.0
angle_radians = np.radians(angle_degrees)

vx = speed * np.cos(angle_radians)
vy = speed * np.sin(angle_radians)

world = World(integrator=EulerCromer())
world.add_force(Gravity(g=9.81))
projectile = world.add(Body(m=1.0, r=(0.0, 0.0), v=(vx, vy), name="Projectile"))

DT = 1.0 / 120.0
log_counter = 0

print("=== RUN STARTED ===")
print(f"Launch configuration: {speed} m/s at {angle_degrees}° under gravity")
print("-" * 83)
print("Time (s) | Pos X (m) | Pos Y (m) |  |V| (m/s) |   KE (J)  |   PE (J)  | Total E (J)")
print("-" * 83)


while world.t == 0.0 or projectile.r[1] > 0.0 or projectile.v[1] > 0.0:
    world.step(DT)
    v_mag = np.linalg.norm(projectile.v)
    if log_counter % 10 == 0:
        print(
            f"{world.t:8.3f} | "
            f"{projectile.r[0]:9.3f} | "
            f"{projectile.r[1]:9.3f} | "
            f"{v_mag:9.3f} | "
            f"{world.kinetic_energy():9.3f} | "
            f"{world.potential_energy():9.3f} | "
            f"{world.total_energy():11.3f}"
        )
    log_counter += 1

projectile.r[1] = 0.0
projectile.v[:] = 0.0

print("-" * 83)
print("TOUCHDOWN SUMMARY:")
print(f"Final Flight Time : {world.t:.3f} s")
print(f"Total Distance X  : {projectile.r[0]:.3f} m")
print(f"Total Energy Check: {world.total_energy():.3f} J (Euler-Cromer Conservation)")
print("=== RUN FINISHED ===")

=== RUN STARTED ===
Launch configuration: 20.0 m/s at 50.0° under gravity
-----------------------------------------------------------------------------------
Time (s) | Pos X (m) | Pos Y (m) |  |V| (m/s) |   KE (J)  |   PE (J)  | Total E (J)
-----------------------------------------------------------------------------------
   0.008 |     0.107 |     0.127 |    19.937 |   198.751 |     1.246 |     199.997
   0.092 |     1.178 |     1.359 |    19.320 |   186.627 |    13.336 |     199.963
   0.175 |     2.250 |     2.524 |    18.717 |   175.171 |    24.758 |     199.930
   0.258 |     3.321 |     3.620 |    18.132 |   164.384 |    35.512 |     199.896
   0.342 |     4.392 |     4.648 |    17.565 |   154.265 |    45.598 |     199.863
   0.425 |     5.464 |     5.608 |    17.019 |   144.815 |    55.015 |     199.830
   0.508 |     6.535 |     6.500 |    16.494 |   136.032 |    63.764 |     199.796
   0.592 |     7.606 |     7.324 |    15.995 |   127.918 |    71.844 |     199.763
   0.675 |